# Fixed-Step Gradient Descent — Interactive Numerical Experiments

Explore fixed-step gradient descent for two quadratic objectives.

Investigate how the step size affects convergence, visualize objective
values and gradient norms, and compare stable and unstable iterations.

**Instructions:** Configure an experiment and click its Run button
to display numerical results and plots.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "gradient_methods").is_dir() and (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Start Jupyter inside the repository checkout.")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from gradient_methods.fixed_step_gradient_solver import solve_fixed_step_gradient


In [ ]:

# Experiment 1 — Fixed-Step Gradient Convergence

fg_objective = widgets.Dropdown(
    options=[("J1 — target (1, ..., 1)", 1),
             ("J2 — target (1, 2, ..., N)", 2)],
    value=1,
    description="Objective",
)

fg_size = widgets.IntSlider(
    value=10, min=2, max=100, step=2, description="Dimension"
)

fg_step = widgets.FloatSlider(
    value=0.25, min=0.05, max=1.5, step=0.05,
    description="Step size",
)

fg_tol = widgets.FloatLogSlider(
    value=1e-8, base=10, min=-12, max=-4,
    step=1, description="Tolerance",
)

fg_budget = widgets.IntSlider(
    value=150, min=10, max=1500, step=10,
    description="Max iterations",
)

fg_button = widgets.Button(
    description="Run experiment",
    button_style="success",
    icon="play",
)

fg_report = widgets.Output()
fg_plots = widgets.Output()


def run_fixed_gradient(_):
    fg_report.clear_output(wait=True)
    fg_plots.clear_output(wait=True)

    n = fg_size.value
    objective = fg_objective.value

    target = (
        np.ones(n)
        if objective == 1
        else np.arange(1, n + 1, dtype=float)
    )

    result = solve_fixed_step_gradient(
        u0=np.zeros(n),
        rho=fg_step.value,
        max_iter=fg_budget.value,
        tol=fg_tol.value,
        objective=objective,
    )

    iterations = np.arange(len(result.cost_history))
    costs = np.asarray(result.cost_history)
    gradients = np.asarray(result.gradient_norms)

    # For the implemented quadratics, J(u) = ||u - u*||_2^2.
    errors = np.sqrt(np.maximum(costs, 0.0))

    with fg_report:
        print(f"Objective: J{objective}")
        print(f"Dimension: {n}")
        print(f"Step size: {fg_step.value:.3f}")
        print(f"Iterations: {result.iterations}")
        print(f"Converged: {result.converged}")
        print(f"Final gradient norm: {gradients[-1]:.6e}")
        print(f"Final solution error: {np.linalg.norm(result.solution - target):.6e}")
        print(f"Termination: {result.message}")

    with fg_plots:
        fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))

        axes[0].semilogy(iterations, np.maximum(costs, 1e-300))
        axes[0].set_title("Objective value")
        axes[0].set_xlabel("Iteration")
        axes[0].set_ylabel("J(u)")
        axes[0].grid(True, alpha=0.3)

        axes[1].semilogy(iterations, np.maximum(gradients, 1e-300))
        axes[1].set_title("Gradient norm")
        axes[1].set_xlabel("Iteration")
        axes[1].set_ylabel("||grad J(u)||")
        axes[1].grid(True, alpha=0.3)

        axes[2].semilogy(iterations, np.maximum(errors, 1e-300))
        axes[2].set_title("Solution error")
        axes[2].set_xlabel("Iteration")
        axes[2].set_ylabel("||u - u*||")
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        display(fig)
        plt.close(fig)


fg_button.on_click(run_fixed_gradient)


In [ ]:

# Experiment 2 — Fixed-Step Size Comparison

sc_objective = widgets.Dropdown(
    options=[
        ("J1 — target (1, ..., 1)", 1),
        ("J2 — target (1, 2, ..., N)", 2),
    ],
    value=1,
    description="Objective",
)

sc_size = widgets.IntSlider(
    value=10, min=2, max=100, step=2,
    description="Dimension",
)

sc_budget = widgets.IntSlider(
    value=40, min=5, max=200, step=5,
    description="Iterations",
)

sc_button = widgets.Button(
    description="Compare step sizes",
    button_style="success",
    icon="play",
)

sc_report = widgets.Output()
sc_plots = widgets.Output()

# Include stable, one-step, oscillating, and divergent examples.
comparison_steps = (0.10, 0.25, 0.50, 0.75, 1.00, 1.10)


def run_step_comparison(_):
    sc_report.clear_output(wait=True)
    sc_plots.clear_output(wait=True)

    n = sc_size.value
    objective = sc_objective.value
    max_iter = sc_budget.value

    results = {}

    for rho in comparison_steps:
        results[rho] = solve_fixed_step_gradient(
            u0=np.zeros(n),
            rho=rho,
            max_iter=max_iter,
            tol=1e-10,
            objective=objective,
        )

    with sc_report:
        print(f"Objective: J{objective}")
        print(f"Dimension: {n}")
        print(f"Maximum iterations: {max_iter}")
        print()
        print(
            f"{'Step size':>10} {'Iterations':>12} "
            f"{'Converged':>12} {'Final gradient':>17}"
        )
        print("-" * 56)

        for rho, result in results.items():
            print(
                f"{rho:10.2f} "
                f"{result.iterations:12d} "
                f"{str(result.converged):>12} "
                f"{result.gradient_norms[-1]:17.6e}"
            )

    with sc_plots:
        fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))

        for rho, result in results.items():
            iterations = np.arange(len(result.cost_history))
            costs = np.asarray(result.cost_history)
            gradients = np.asarray(result.gradient_norms)

            axes[0].semilogy(
                iterations,
                np.maximum(costs, 1e-14),
                label=f"rho = {rho:.2f}",
            )

            axes[1].semilogy(
                iterations,
                np.maximum(gradients, 1e-14),
                label=f"rho = {rho:.2f}",
            )

        axes[0].set_title("Objective value comparison")
        axes[0].set_xlabel("Iteration")
        axes[0].set_ylabel("J(u)")
        axes[0].grid(True, alpha=0.3)

        axes[1].set_title("Gradient norm comparison")
        axes[1].set_xlabel("Iteration")
        axes[1].set_ylabel("||grad J(u)||")
        axes[1].grid(True, alpha=0.3)

        for ax in axes:
            ax.legend(fontsize=8)

        plt.tight_layout()
        display(fig)
        plt.close(fig)


sc_button.on_click(run_step_comparison)


In [ ]:

# -------------------------------------------------------
# Fixed-Step Gradient Descent — Compact Voilà dashboard
# -------------------------------------------------------

fg_status = widgets.HTML()
sc_status = widgets.HTML()

fg_parameters = (
    fg_objective, fg_size, fg_step, fg_tol, fg_budget
)
sc_parameters = (
    sc_objective, sc_size, sc_budget
)

last_runs = {"gradient": None, "comparison": None}


def parameter_snapshot(parameters):
    return tuple(widget.value for widget in parameters)


def update_status(key, parameters, indicator):
    current = parameter_snapshot(parameters)
    previous = last_runs[key]

    if previous is None:
        label = "Not run yet"
        color = "#64748b"
    elif current == previous:
        label = "Simulation up to date"
        color = "#15803d"
    else:
        label = "Parameters changed — run again"
        color = "#c2410c"

    indicator.value = (
        f"<span style='font-weight:600;color:{color};'>"
        f"● {label}</span>"
    )


def mark_gradient_complete(change=None):
    last_runs["gradient"] = parameter_snapshot(fg_parameters)
    update_status("gradient", fg_parameters, fg_status)


def mark_comparison_complete(change=None):
    last_runs["comparison"] = parameter_snapshot(sc_parameters)
    update_status("comparison", sc_parameters, sc_status)


# Existing numerical callbacks execute first. These callbacks
# update the displayed simulation-status indicators afterward.
fg_button.on_click(mark_gradient_complete)
sc_button.on_click(mark_comparison_complete)

for control in fg_parameters:
    control.observe(
        lambda change: update_status(
            "gradient", fg_parameters, fg_status
        ),
        names="value",
    )

for control in sc_parameters:
    control.observe(
        lambda change: update_status(
            "comparison", sc_parameters, sc_status
        ),
        names="value",
    )


report_layout = widgets.Layout(
    width="100%",
    padding="6px 4px",
)

fg_report_box = widgets.Accordion(
    children=[fg_report],
    selected_index=None,
    layout=report_layout,
)
fg_report_box.set_title(0, "Numerical report")

sc_report_box = widgets.Accordion(
    children=[sc_report],
    selected_index=None,
    layout=report_layout,
)
sc_report_box.set_title(0, "Numerical report")


def parameter_panel(title, description, controls, button, status):
    title_html = widgets.HTML(
        f"<h3 style='margin:0 0 6px 0;'>{title}</h3>"
        f"<p style='margin:0 0 12px 0;color:#475569;'>"
        f"{description}</p>"
    )

    return widgets.VBox(
        [title_html, *controls, button, status],
        layout=widgets.Layout(
            width="100%",
            padding="16px",
            border="1px solid #dbe3ed",
        ),
    )


gradient_panel = parameter_panel(
    "Gradient Convergence",
    "Explore the convergence of a selected quadratic objective.",
    fg_parameters,
    fg_button,
    fg_status,
)

comparison_panel = parameter_panel(
    "Step-Size Comparison",
    "Compare stable, oscillating, and divergent step sizes.",
    sc_parameters,
    sc_button,
    sc_status,
)


controls_row = widgets.HBox(
    [gradient_panel, comparison_panel],
    layout=widgets.Layout(
        width="100%",
        gap="16px",
        align_items="stretch",
    ),
)

gradient_results = widgets.VBox(
    [
        widgets.HTML("<h3>Gradient Convergence Results</h3>"),
        fg_plots,
        fg_report_box,
    ],
    layout=widgets.Layout(width="100%"),
)

comparison_results = widgets.VBox(
    [
        widgets.HTML("<h3>Step-Size Comparison Results</h3>"),
        sc_plots,
        sc_report_box,
    ],
    layout=widgets.Layout(width="100%"),
)

# Keep results hidden until a user launches the experiment.
gradient_results.layout.display = "none"
comparison_results.layout.display = "none"


def show_gradient_results(button):
    gradient_results.layout.display = ""


def show_comparison_results(button):
    comparison_results.layout.display = ""


fg_button.on_click(show_gradient_results)
sc_button.on_click(show_comparison_results)


# Five plots in one horizontal row: three + two.
gradient_results.layout.width = "60%"
comparison_results.layout.width = "40%"

results_row = widgets.HBox(
    [gradient_results, comparison_results],
    layout=widgets.Layout(
        width="100%",
        align_items="flex-start",
        flex_flow="row",
        overflow_x="auto",
    ),
)

dashboard = widgets.VBox(
    [
        controls_row,
        results_row,
    ],
    layout=widgets.Layout(width="100%"),
)

update_status("gradient", fg_parameters, fg_status)
update_status("comparison", sc_parameters, sc_status)

display(dashboard)
